# Session 4 — GroupBy & Merging

Runnable code for the **Build It** and **Experiment** sections.
Run each cell top to bottom.

In [1]:
import numpy as np
import pandas as pd

## 1. Build It — Core Code

### 1.1 A tiny sales table

In [2]:
sales = pd.DataFrame({
    "region":  ["North", "South", "North", "South", "North", "South"],
    "product": ["Laptop", "Phone", "Phone", "Laptop", "Tablet", "Tablet"],
    "revenue": [1200, 800, 400, 1500, 500, 900],
})

sales

,region,product,revenue
0,North,Laptop,1200
1,South,Phone,800
2,North,Phone,400
3,South,Laptop,1500
4,North,Tablet,500
5,South,Tablet,900


### 4.1 Split–apply–combine with `groupby`

In [3]:
sales.groupby("region")["revenue"].sum()

region
North    2100
South    3200
Name: revenue, dtype: int64

In [4]:
sales.groupby(["region", "product"])["revenue"].sum()

region  product
North   Laptop     1200
        Phone       400
        Tablet      500
South   Laptop     1500
        Phone       800
        Tablet      900
Name: revenue, dtype: int64

### 4.2 Aggregation with `.agg()` — lists, dictionaries, and named aggregation

In [5]:
sales.groupby("region")["revenue"].agg(["sum", "mean", "max"])

,sum,mean,max
region,,,
North,2100,700.000000,1200
South,3200,1066.666667,1500


In [6]:
sales.groupby("region").agg(
    total=("revenue", "sum"),
    avg=("revenue", "mean"),
    orders=("product", "count"),
)

,total,avg,orders
region,,,
North,2100,700.000000,3
South,3200,1066.666667,3


### 4.3 `transform` — broadcast a group value back to every row

In [7]:
sales = sales.assign(
    region_total=lambda d: d.groupby("region")["revenue"].transform("sum"),
)
sales["share_of_region"] = sales["revenue"] / sales["region_total"]
sales

,region,product,revenue,region_total,share_of_region
0,North,Laptop,1200,2100,0.571429
1,South,Phone,800,3200,0.250000
2,North,Phone,400,2100,0.190476
3,South,Laptop,1500,3200,0.468750
4,North,Tablet,500,2100,0.238095
5,South,Tablet,900,3200,0.281250


### 4.4 `value_counts` — count categories

In [8]:
sales["region"].value_counts()

region
North    3
South    3
Name: count, dtype: int64

In [9]:
sales["product"].value_counts(normalize=True)

product
Laptop    0.333333
Phone     0.333333
Tablet    0.333333
Name: proportion, dtype: float64

### 4.5 `crosstab` — count combinations of two columns

In [10]:
pd.crosstab(sales["region"], sales["product"], margins=True)

product,Laptop,Phone,Tablet,All
region,,,,
North,1,1,1,3
South,1,1,1,3
All,2,2,2,6


In [11]:
pd.crosstab(
    sales["region"],
    sales["product"],
    values=sales["revenue"],
    aggfunc="sum",
    margins=True,
)

product,Laptop,Phone,Tablet,All
region,,,,
North,1200,400,500,2100
South,1500,800,900,3200
All,2700,1200,1400,5300


### 4.6 `pivot_table` — reshape long data into a matrix

In [12]:
sales.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)

product,Laptop,Phone,Tablet,All
region,,,,
North,1200,400,500,2100
South,1500,800,900,3200
All,2700,1200,1400,5300


### 4.7 `merge` — combine two tables on a shared key

In [13]:
customers = pd.DataFrame({
    "customer_id": [101, 102, 103, 104, 105, 106],
    "name": ["Alice", "Bob", "Carol", "Dan", "Eve", "Frank"],
    "region": ["North", "South", "North", "East", "West", "South"],
})

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8],
    "customer_id": [101, 102, 101, 103, 104, 102, 105, 103],
    "product": ["Laptop", "Phone", "Tablet", "Laptop", "Phone", "Laptop", "Tablet", "Tablet"],
    "quantity": [1, 2, 1, 1, 3, 1, 2, 1],
    "unit_price": [1200, 800, 500, 1200, 800, 1200, 500, 500],
})

orders = orders.assign(revenue=orders["quantity"] * orders["unit_price"])
orders

,order_id,customer_id,product,quantity,unit_price,revenue
0,1,101,Laptop,1,1200,1200
1,2,102,Phone,2,800,1600
2,3,101,Tablet,1,500,500
3,4,103,Laptop,1,1200,1200
4,5,104,Phone,3,800,2400
5,6,102,Laptop,1,1200,1200
6,7,105,Tablet,2,500,1000
7,8,103,Tablet,1,500,500


In [14]:
# how="inner": only customer_ids present in both tables (106 is dropped)
orders.merge(customers, on="customer_id", how="inner")

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1,101,Laptop,1,1200,1200,Alice,North
1,2,102,Phone,2,800,1600,Bob,South
2,3,101,Tablet,1,500,500,Alice,North
3,4,103,Laptop,1,1200,1200,Carol,North
4,5,104,Phone,3,800,2400,Dan,East
5,6,102,Laptop,1,1200,1200,Bob,South
6,7,105,Tablet,2,500,1000,Eve,West
7,8,103,Tablet,1,500,500,Carol,North


In [15]:
# how="left": keep every order row
orders.merge(customers, on="customer_id", how="left")

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1,101,Laptop,1,1200,1200,Alice,North
1,2,102,Phone,2,800,1600,Bob,South
2,3,101,Tablet,1,500,500,Alice,North
3,4,103,Laptop,1,1200,1200,Carol,North
4,5,104,Phone,3,800,2400,Dan,East
5,6,102,Laptop,1,1200,1200,Bob,South
6,7,105,Tablet,2,500,1000,Eve,West
7,8,103,Tablet,1,500,500,Carol,North


In [16]:
# how="right": keep every customer, including Frank (106) who has no orders
orders.merge(customers, on="customer_id", how="right")

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1.0,101,Laptop,1.0,1200.0,1200.0,Alice,North
1,3.0,101,Tablet,1.0,500.0,500.0,Alice,North
2,2.0,102,Phone,2.0,800.0,1600.0,Bob,South
3,6.0,102,Laptop,1.0,1200.0,1200.0,Bob,South
4,4.0,103,Laptop,1.0,1200.0,1200.0,Carol,North
5,8.0,103,Tablet,1.0,500.0,500.0,Carol,North
6,5.0,104,Phone,3.0,800.0,2400.0,Dan,East
7,7.0,105,Tablet,2.0,500.0,1000.0,Eve,West
8,NaN,106,NaN,NaN,NaN,NaN,Frank,South


In [17]:
# how="outer": keep every key from both sides
orders.merge(customers, on="customer_id", how="outer")

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1.0,101,Laptop,1.0,1200.0,1200.0,Alice,North
1,3.0,101,Tablet,1.0,500.0,500.0,Alice,North
2,2.0,102,Phone,2.0,800.0,1600.0,Bob,South
3,6.0,102,Laptop,1.0,1200.0,1200.0,Bob,South
4,4.0,103,Laptop,1.0,1200.0,1200.0,Carol,North
5,8.0,103,Tablet,1.0,500.0,500.0,Carol,North
6,5.0,104,Phone,3.0,800.0,2400.0,Dan,East
7,7.0,105,Tablet,2.0,500.0,1000.0,Eve,West
8,NaN,106,NaN,NaN,NaN,NaN,Frank,South


### 4.8 `join` (on the index) and `concat` (stack tables)

In [18]:
by_region = (
    orders.merge(customers, on="customer_id", how="left")
          .groupby("region")[["revenue"]]
          .sum()
)
targets = pd.DataFrame(
    {"target": [2000, 1500, 1200, 1000, 500]},
    index=["East", "North", "South", "West", "Central"],
)
by_region.join(targets, how="left")

,revenue,target
region,,
East,2400,2000
North,3400,1500
South,2800,1200
West,1000,1000


In [19]:
q1 = pd.DataFrame({"month": ["Jan", "Feb"], "sales": [100, 200]})
q2 = pd.DataFrame({"month": ["Mar", "Apr"], "sales": [300, 400]})
pd.concat([q1, q2], ignore_index=True)

,month,sales
0,Jan,100
1,Feb,200
2,Mar,300
3,Apr,400


In [20]:
left = pd.DataFrame({"id": [1, 2, 3], "score": [10, 20, 30]})
right = pd.DataFrame({"id": [1, 2, 3], "grade": ["A", "B", "C"]})
pd.concat([left, right], axis=1)

,id,score,id,grade
0,1,10,1,A
1,2,20,2,B
2,3,30,3,C


## 2. Experiment

Change a parameter, predict the output, then run the cell.

In [21]:
# Experiment 1: group by two keys -> a MultiIndex is returned
sales.groupby(["region", "product"])["revenue"].agg(["sum", "count"])

sum  count
region product             
North  Laptop   1200      1
       Phone     400      1
       Tablet    500      1
South  Laptop   1500      1
       Phone     800      1
       Tablet    900      1

In [22]:
# Experiment 2: agg collapses rows, transform keeps them
agg_result = sales.groupby("region")["revenue"].sum()
transform_result = sales.groupby("region")["revenue"].transform("sum")
print("agg shape:      ", agg_result.shape)
print("transform shape:", transform_result.shape)

agg shape:       (2,)
transform shape: (6,)


In [23]:
# Experiment 3: normalize a crosstab to row proportions
pd.crosstab(sales["region"], sales["product"], normalize="index").round(2)

product,Laptop,Phone,Tablet
region,,,
North,0.33,0.33,0.33
South,0.33,0.33,0.33


In [24]:
# Experiment 4: pivot_table averages instead of sums
sales.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="mean",
    fill_value=0,
)

product,Laptop,Phone,Tablet
region,,,
North,1200.0,400.0,500.0
South,1500.0,800.0,900.0


In [25]:
# Experiment 5: suffixes disambiguate overlapping column names
a = pd.DataFrame({"id": [1, 2], "value": [10, 20]})
b = pd.DataFrame({"id": [1, 2], "value": [100, 200]})
a.merge(b, on="id", suffixes=("_a", "_b"))

,id,value_a,value_b
0,1,10,100
1,2,20,200


## 3. Mini Project — Orders + Customers

Combine orders with customers, then summarize sales by region.

In [26]:
# Step 1: merge every order with its customer's region
merged = orders.merge(customers, on="customer_id", how="left")
merged

,order_id,customer_id,product,quantity,unit_price,revenue,name,region
0,1,101,Laptop,1,1200,1200,Alice,North
1,2,102,Phone,2,800,1600,Bob,South
2,3,101,Tablet,1,500,500,Alice,North
3,4,103,Laptop,1,1200,1200,Carol,North
4,5,104,Phone,3,800,2400,Dan,East
5,6,102,Laptop,1,1200,1200,Bob,South
6,7,105,Tablet,2,500,1000,Eve,West
7,8,103,Tablet,1,500,500,Carol,North


In [27]:
# Step 2: summarize each region with named aggregation
by_region = merged.groupby("region").agg(
    total_revenue=("revenue", "sum"),
    orders=("order_id", "count"),
    avg_order=("revenue", "mean"),
)
by_region

,total_revenue,orders,avg_order
region,,,
East,2400,1,2400.0
North,3400,4,850.0
South,2800,2,1400.0
West,1000,1,1000.0


In [28]:
# Step 3: which products drive each region?
merged.pivot_table(
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0,
    margins=True,
)

product,Laptop,Phone,Tablet,All
region,,,,
East,0,2400,0,2400
North,2400,0,1000,3400
South,1200,1600,0,2800
West,0,0,1000,1000
All,3600,4000,2000,9600


In [29]:
# Step 4: rank the regions by revenue
by_region.sort_values("total_revenue", ascending=False)

,total_revenue,orders,avg_order
region,,,
North,3400,4,850.0
South,2800,2,1400.0
East,2400,1,2400.0
West,1000,1,1000.0
